# Lab D — Desplegar tu modelo custom con LoRA dinámico en un Vertex AI Endpoint

Este notebook es la segunda variante del Lab C (`class04c-deploy-merged.ipynb`):
en vez de fusionar el adaptador LoRA dentro del modelo base, desplegamos el
modelo **base sin tocar** (`google/gemma-7b-it`) y le decimos a vLLM que
active soporte de LoRA (`--enable-lora`). El adaptador se pasa **en cada
petición de predicción** como un campo `dynamic-lora` apuntando directamente
al bucket de Cloud Storage -sin fusionar nada, y sin necesidad de GPU local
para preparar el modelo (a diferencia del Lab C, aquí no hace falta cargar
ni un solo peso del modelo en esta VM).

## ¿Por qué esta variante y no la fusionada?

- **Un solo despliegue puede servir varios adaptadores.** Si en el curso
  entrenan distintos adaptadores (por ejemplo, uno por grupo, o uno por
  dataset), no hace falta desplegar un endpoint por cada uno -todos comparten
  el mismo modelo base ya cargado en GPU, y vLLM intercambia el adaptador
  activo por petición (controlado por `max_loras`/`max_cpu_loras`).
- **No hay que fusionar ni volver a subir ~14GB por adaptador** -el
  adaptador LoRA es liviano (unos pocos MB-cientos de MB), así que iterar
  sobre nuevas versiones del fine-tuning es mucho más rápido: solo hay que
  volver a entrenar y apuntar `dynamic-lora` al nuevo bucket, sin tocar el
  despliegue del modelo base.
- **Contra:** el modelo base tiene que poder descargarse dentro del
  contenedor de Vertex AI en el momento del despliegue -como Gemma es
  *gated* en Hugging Face, hay que pasarle un token de HF al contenedor (ver
  paso 5), algo que el Lab C no necesita (porque el modelo fusionado ya está
  materializado en tu propio bucket, sin gating).

## Arquitectura de este lab

```
google/gemma-7b-it  (modelo base, se descarga de HF dentro del contenedor)
        │
        ▼
  aiplatform.Model.upload(..., enable_lora=True)  -> Vertex AI Model Registry
        │
        ▼
  aiplatform.Endpoint.create() + model.deploy()   -> Endpoint con GPU (vLLM)
        │
        ▼
  endpoint.predict(instances=[{..., "dynamic-lora": "gs://bucket/gemma-7b-it-samsum-lora"}])
        │
        ▼
  resúmenes generados CON el adaptador (o sin él, si se omite dynamic-lora)
```

**Requisito:** el mismo que en el Lab C -tener la ruta `gs://.../gemma-7b-it-samsum-lora`
del Custom Training Job (o del entrenamiento en la VM, ya subido a GCS).

A diferencia del Lab C, este notebook **no necesita GPU local**: no cargamos
el modelo en ningún momento aquí, solo el tokenizer (liviano) para construir
el prompt. Puede correr en el mismo contenedor JupyterHub o en cualquier
entorno con el SDK de Vertex AI instalado.


## 0. Instalar dependencias

In [ ]:
%pip install --quiet transformers huggingface_hub google-cloud-storage google-cloud-aiplatform


## 1. Configuración


In [ ]:
import os

PROJECT_ID = "[tu-proyecto-gcp]"          # <-- reemplaza esto
LOCATION = "us-central1"

BASE_MODEL_ID = "google/gemma-7b-it"      # se descarga de Hugging Face dentro del contenedor
GCS_ADAPTER_DIR = "gs://[tu-bucket]/gemma-7b-it-samsum-lora"   # <-- salida del entrenamiento

LOCAL_ADAPTER_DIR = "/home/jovyan/labs/gemma-7b-it-samsum-lora-vertex"

MACHINE_TYPE = "g2-standard-4"
ACCELERATOR_TYPE = "NVIDIA_L4"
ACCELERATOR_COUNT = 1

SERVICE_ACCOUNT = None   # opcional, cuenta de servicio con permiso sobre el bucket del adaptador

for v in (PROJECT_ID, GCS_ADAPTER_DIR):
    assert not v.startswith("["), f"Falta reemplazar un placeholder: {v!r}"


## 2. Descargar SOLO el tokenizer del adaptador

No necesitamos el modelo completo aquí -el tokenizer (unos pocos KB de
archivos JSON) es suficiente para construir el prompt con el formato de chat
de Gemma, igual que en `infer_gemma_vertex.py`. El modelo real lo carga vLLM
directamente dentro del contenedor desplegado, no en esta VM.


In [ ]:
def download_gcs_dir(gcs_uri, local_dir, force=False):
    if os.path.isdir(local_dir) and os.listdir(local_dir) and not force:
        print(f"Ya existe una copia local en {local_dir}. Saltando descarga.")
        return local_dir

    from google.cloud import storage

    bucket_name, _, prefix = gcs_uri[len("gs://"):].partition("/")
    prefix = prefix.rstrip("/")

    print(f"Descargando {gcs_uri} -> {local_dir} ...")
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blobs = [b for b in bucket.list_blobs(prefix=prefix + "/" if prefix else prefix)
             if not b.name.endswith("/")]

    if not blobs:
        raise SystemExit(f"No encontré archivos en {gcs_uri}. ¿Es correcta la ruta?")

    os.makedirs(local_dir, exist_ok=True)
    for blob in blobs:
        rel_path = blob.name[len(prefix):].lstrip("/") if prefix else blob.name
        dest = os.path.join(local_dir, rel_path)
        os.makedirs(os.path.dirname(dest) or local_dir, exist_ok=True)
        blob.download_to_filename(dest)

    print(f"Descarga completa: {len(blobs)} archivo(s) en {local_dir}")
    return local_dir


ADAPTER_DIR = download_gcs_dir(GCS_ADAPTER_DIR, LOCAL_ADAPTER_DIR)

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
print("Tokenizer cargado desde:", ADAPTER_DIR)


## 3. Token de Hugging Face para el contenedor del endpoint

A diferencia del Lab C (donde el modelo fusionado ya vive en tu propio
bucket, sin gating), aquí vLLM va a descargar `google/gemma-7b-it`
**dentro del contenedor de Vertex AI** en el momento del despliegue -y ese
modelo es *gated*. Le pasamos el token como variable de entorno del
contenedor (`serving_container_environment_variables`), nunca hardcodeado en
el código.


In [ ]:
import getpass

hf_token = getpass.getpass("Token de Hugging Face (con acceso a google/gemma-7b-it): ")


## 4. Inicializar el SDK de Vertex AI


In [ ]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=LOCATION)
print(f"Vertex AI inicializado: proyecto={PROJECT_ID}, región={LOCATION}")


## 5. El contenedor vLLM prebuilt de Vertex AI

Mismo contenedor que en el Lab C -revisa la versión más reciente en la
documentación de Model Garden antes de usar esto en producción.


In [ ]:
VLLM_DOCKER_URI = (
    "us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/"
    "pytorch-vllm-serve:20241210_0916_RC00"
)


## 6. Función de despliegue (idéntica a la del Lab C)

Reutilizamos exactamente el mismo helper -la única diferencia real está en
qué le pasamos: `model_id` es ahora el repo de Hugging Face (no una ruta de
GCS), `enable_lora=True`, y le sumamos el token de HF como variable de
entorno del contenedor.


In [ ]:
from typing import Optional, Tuple


def deploy_model_vllm(
    model_name: str,
    model_id: str,
    machine_type: str = MACHINE_TYPE,
    accelerator_type: str = ACCELERATOR_TYPE,
    accelerator_count: int = ACCELERATOR_COUNT,
    gpu_memory_utilization: float = 0.85,
    max_model_len: int = 2048,
    dtype: str = "bfloat16",
    enable_lora: bool = False,
    max_loras: int = 1,
    max_cpu_loras: int = 4,
    env_vars: Optional[dict] = None,
    service_account: Optional[str] = None,
) -> Tuple["aiplatform.Model", "aiplatform.Endpoint"]:
    vllm_args = [
        "python", "-m", "vllm.entrypoints.api_server",
        "--host=0.0.0.0", "--port=8080",
        f"--model={model_id}",
        f"--tensor-parallel-size={accelerator_count}",
        "--swap-space=16",
        f"--gpu-memory-utilization={gpu_memory_utilization}",
        f"--max-model-len={max_model_len}",
        f"--dtype={dtype}",
        f"--max-loras={max_loras}",
        f"--max-cpu-loras={max_cpu_loras}",
        "--disable-log-stats",
    ]
    if enable_lora:
        vllm_args.append("--enable-lora")

    model = aiplatform.Model.upload(
        display_name=model_name,
        serving_container_image_uri=VLLM_DOCKER_URI,
        serving_container_args=vllm_args,
        serving_container_ports=[8080],
        serving_container_predict_route="/generate",
        serving_container_health_route="/ping",
        serving_container_environment_variables=env_vars or {},
        serving_container_shared_memory_size_mb=16 * 1024,
        serving_container_deployment_timeout=7200,
    )
    print("Modelo registrado:", model.resource_name)

    endpoint = aiplatform.Endpoint.create(display_name=f"{model_name}-endpoint")
    print("Endpoint creado:", endpoint.resource_name)

    print("Desplegando (esto puede tardar 10-20 minutos: descarga del modelo + arranque de vLLM)...")
    model.deploy(
        endpoint=endpoint,
        machine_type=machine_type,
        accelerator_type=accelerator_type,
        accelerator_count=accelerator_count,
        deploy_request_timeout=1800,
        service_account=service_account,
    )
    print("Despliegue completo.")
    return model, endpoint


## 7. Desplegar el modelo BASE con soporte de LoRA dinámico

`max_loras=2` (en vez de 1) deja lugar para un segundo adaptador en el
ejercicio, sin tener que volver a desplegar.


In [ ]:
modelo_vertex, endpoint = deploy_model_vllm(
    model_name="gemma-7b-it-lora-dinamico",
    model_id=BASE_MODEL_ID,
    enable_lora=True,
    max_loras=2,
    env_vars={"HF_TOKEN": hf_token},
    service_account=SERVICE_ACCOUNT,
)

del hf_token   # ya se usó para desplegar, no lo dejamos flotando en una variable más de lo necesario


## 8. Probar el endpoint: con adaptador y sin adaptador

Mismo diálogo, misma petición -la única diferencia es si mandamos
`dynamic-lora` o no. Esto es el equivalente, ya contra un endpoint remoto, del
flag `--compare_base`/`disable_adapter()` de `infer_gemma.py`.


In [ ]:
def build_prompt(dialogue):
    messages = [
        {"role": "user", "content": f"Resume el siguiente diálogo en 1-2 frases:\n\n{dialogue}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def resumir_endpoint(dialogue, lora_id=None, max_tokens=64, temperature=0.0):
    prompt = build_prompt(dialogue)
    instance = {
        "prompt": prompt,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "top_p": 1.0,
        "top_k": -1,
    }
    if lora_id:
        instance["dynamic-lora"] = lora_id
    response = endpoint.predict(instances=[instance])
    return response.predictions[0]


In [ ]:
dialogo_prueba = (
    "Carlos: ¿Vas a venir a la reunión de las 3pm?\n"
    "Marta: Sí, ya salgo. ¿La sala sigue siendo la 402?\n"
    "Carlos: Sí, misma sala. Nos vemos ahí."
)

print("Diálogo:\n" + dialogo_prueba)
print("\n>> Con LoRA dinámico (afinado, adaptador desde GCS):\n"
      + resumir_endpoint(dialogo_prueba, lora_id=GCS_ADAPTER_DIR))
print("\n>> Sin LoRA (modelo base, sin adaptador):\n"
      + resumir_endpoint(dialogo_prueba, lora_id=None))


## 9. Varios ejemplos con el adaptador activo


In [ ]:
ejemplos = [
    "Sofía: Se me quedó el cargador en tu casa ayer, ¿lo tienes ahí?\n"
    "Diego: Sí, lo vi en la mesa de la sala. Te lo llevo mañana a la oficina.\n"
    "Sofía: Perfecto, gracias!",

    "Ana: hey are we still on for the gym at 6?\n"
    "Leo: yeah but running a bit late, more like 6:20\n"
    "Ana: no worries, I'll grab a locker and wait",
]

for i, dialogo in enumerate(ejemplos, start=1):
    print(f"\n{'=' * 70}\nEjemplo {i}\n{'=' * 70}")
    print("Diálogo:\n" + dialogo)
    print("\n>> Resumen (LoRA dinámico, GCS_ADAPTER_DIR):\n"
          + resumir_endpoint(dialogo, lora_id=GCS_ADAPTER_DIR))


## 10. Servir un segundo adaptador (sin redeploy)

Si entrenaran un segundo adaptador -por ejemplo sobre otro dataset o con
otros hiperparámetros- y lo subieran a otra carpeta de GCS, podrían probarlo
contra este MISMO endpoint ya desplegado, solo cambiando `dynamic-lora`:

```python
GCS_ADAPTER_DIR_V2 = "gs://[tu-bucket]/gemma-7b-it-otro-dataset-lora"
resumir_endpoint(dialogo_prueba, lora_id=GCS_ADAPTER_DIR_V2)
```

`max_cpu_loras` (definido en el paso 7) controla cuántos adaptadores puede
mantener vLLM cacheados en CPU listos para intercambiar rápido -si superan
ese número, el siguiente adaptador nuevo desplaza al menos usado.


## 11. Costos y buenas prácticas

- Igual que en el Lab C, la GPU se cobra por hora mientras el modelo esté
  desplegado en el endpoint.
- La ventaja de costo de esta variante aparece cuando sirven **más de un**
  adaptador: un solo endpoint (una sola GPU pagada) puede atender N
  adaptadores distintos, en vez de N endpoints (N GPUs) como exigiría la
  variante fusionada del Lab C.
- Contrapartida de latencia: cada intercambio de adaptador tiene un costo
  pequeño de carga la primera vez que se usa (luego queda cacheado según
  `max_cpu_loras`) -para un solo adaptador fijo y de alto volumen, fusionar
  (Lab C) sigue siendo lo más simple y predecible.


## 12. Limpieza obligatoria

In [ ]:
endpoint.undeploy_all()
endpoint.delete()
modelo_vertex.delete()
print("Endpoint y modelo eliminados. Ya no se cobra por este despliegue.")


## Ejercicio

1. Entrena (o simula, reusando el mismo adaptador con otro nombre de
   carpeta en GCS) un segundo adaptador y pruébalo contra el mismo endpoint
   sin redesplegar, como en la sección 10.
2. Mide la latencia de la primera llamada con un adaptador nuevo vs. las
   llamadas siguientes con el mismo adaptador (¿ves el efecto de
   `max_cpu_loras`?).
3. Compara el costo de este Lab D (un endpoint sirviendo 2 adaptadores)
   contra el de desplegar el Lab C dos veces (uno por adaptador fusionado).
4. Reescribe `evaluate_gemma_vertex.py` para que en vez de cargar el modelo
   localmente, llame a `endpoint.predict(...)` de este lab -¿los resultados
   de ROUGE/BERTScore cambian frente a la evaluación local? (deberían ser
   iguales o casi iguales; si no, revisa que el prompt enviado al endpoint
   sea exactamente el mismo `build_prompt` usado localmente).
5. Discute con tu grupo: si el proyecto final necesita servir un modelo
   afinado por cada equipo/usuario, ¿qué variante (fusionado o LoRA
   dinámico) escalaría mejor en costo? ¿Por qué?

## Notas finales

- El equivalente sin notebook (usando solo `gcloud`) está en
  `class04d-deploy-gcloud.sh`.
- Ver `class04c-deploy-merged.ipynb` para la variante que fusiona los pesos
  -más simple si solo van a servir un adaptador fijo, sin necesidad de
  intercambiarlo dinámicamente.
